In [1]:
import pandas as pd
from pathlib import Path
from os import getcwd
import numpy as np

In [2]:
ROOT_DIR = Path(getcwd()).parent.parent
RAW_DIR = ROOT_DIR / "data/raw"
INTERIM_DIR = ROOT_DIR / "data/interim"
PROCESSED_DIR = ROOT_DIR / "data/processed"

# 1. SP100 Data

In [3]:
df = pd.read_parquet(RAW_DIR / "prices_raw.parquet")
print(df.head())

        date ticker   adj_close       close        high         low  \
0 2021-01-04   AAPL  125.856697  129.410004  133.610001  126.760002   
1 2021-01-05   AAPL  127.412758  131.009995  131.740005  128.429993   
2 2021-01-06   AAPL  123.123840  126.599998  131.050003  126.379997   
3 2021-01-07   AAPL  127.325241  130.919998  131.630005  127.860001   
4 2021-01-08   AAPL  128.424240  132.050003  132.630005  130.229996   

         open     volume  
0  133.520004  143301900  
1  128.889999   97664900  
2  127.720001  155088000  
3  128.360001  109578200  
4  132.429993  105158200  


In [4]:
# Columns types
print(df.dtypes)

date         datetime64[ms]
ticker                  str
adj_close           float64
close               float64
high                float64
low                 float64
open                float64
volume                int64
dtype: object


# 2. Metadata Exploration

In [5]:
metadata = pd.read_parquet(RAW_DIR / "ticker_metadata.parquet")
print(metadata.head())

  ticker      sector                         industry     market_cap  \
0   AAPL  Technology             Consumer Electronics  3776182222848   
1   ABBV  Healthcare     Drug Manufacturers - General   360473788416   
2    ABT  Healthcare                  Medical Devices   172512804864   
3    ACN  Technology  Information Technology Services   114569404416   
4   ADBE  Technology           Software - Application    96137871360   

            short_name currency  
0           Apple Inc.      USD  
1          AbbVie Inc.      USD  
2  Abbott Laboratories      USD  
3        Accenture plc      USD  
4           Adobe Inc.      USD  


In [6]:
# What columns are in the metadata?
print(metadata.columns)

Index(['ticker', 'sector', 'industry', 'market_cap', 'short_name', 'currency'], dtype='str')


In [7]:
print(metadata.dtypes)

ticker          str
sector          str
industry        str
market_cap    int64
short_name      str
currency        str
dtype: object


In [8]:
# Unique sectors and number of tickers in each sector
sector_counts = metadata["sector"].value_counts()
print(sector_counts)
print(f"The number of sectors is: {metadata['sector'].nunique()}")

sector
Financial Services        16
Healthcare                14
Technology                12
Industrials               12
Consumer Defensive        10
Consumer Cyclical          9
Communication Services     7
Energy                     3
Utilities                  3
Real Estate                2
Name: count, dtype: int64
The number of sectors is: 10


In [9]:
# Unique industries and number of tickers in each industry
industry_counts = metadata["industry"].value_counts()
print(industry_counts)

industry
Drug Manufacturers - General           8
Credit Services                        5
Aerospace & Defense                    5
Banks - Diversified                    5
Semiconductors                         4
Telecom Services                       3
Discount Stores                        3
Utilities - Regulated Electric         3
Medical Devices                        2
Information Technology Services        2
Software - Application                 2
Household & Personal Products          2
Healthcare Plans                       2
Oil & Gas Integrated                   2
Diagnostics & Research                 2
Entertainment                          2
Integrated Freight & Logistics         2
Auto Manufacturers                     2
Internet Content & Information         2
Capital Markets                        2
Home Improvement Retail                2
Conglomerates                          2
Beverages - Non-Alcoholic              2
Restaurants                            2
Tobacco

In [10]:
# Are all tickers traded using the same currency?
currency_counts = metadata["currency"].value_counts()
print(currency_counts)

currency
USD    88
Name: count, dtype: int64


# 3. Base Panel

In [11]:
interim = pd.read_parquet(INTERIM_DIR / "base_panel.parquet")
print(interim.head())

        date ticker   adj_close       close        high         low  \
0 2021-01-04   AAPL  125.856697  129.410004  133.610001  126.760002   
1 2021-01-05   AAPL  127.412758  131.009995  131.740005  128.429993   
2 2021-01-06   AAPL  123.123840  126.599998  131.050003  126.379997   
3 2021-01-07   AAPL  127.325241  130.919998  131.630005  127.860001   
4 2021-01-08   AAPL  128.424240  132.050003  132.630005  130.229996   

         open     volume      sector              industry     market_cap  \
0  133.520004  143301900  Technology  Consumer Electronics  3776182222848   
1  128.889999   97664900  Technology  Consumer Electronics  3776182222848   
2  127.720001  155088000  Technology  Consumer Electronics  3776182222848   
3  128.360001  109578200  Technology  Consumer Electronics  3776182222848   
4  132.429993  105158200  Technology  Consumer Electronics  3776182222848   

   short_name currency  
0  Apple Inc.      USD  
1  Apple Inc.      USD  
2  Apple Inc.      USD  
3  Apple I

In [12]:
print(interim.dtypes)

date          datetime64[ms]
ticker                   str
adj_close            float64
close                float64
high                 float64
low                  float64
open                 float64
volume                 int64
sector                   str
industry                 str
market_cap             int64
short_name               str
currency                 str
dtype: object


# 4. Features

In [13]:
features = pd.read_parquet(INTERIM_DIR / "features_panel.parquet")
print(features.head())

        date ticker   adj_close       close        high         low  \
0 2021-01-04   AAPL  125.856697  129.410004  133.610001  126.760002   
1 2021-01-05   AAPL  127.412758  131.009995  131.740005  128.429993   
2 2021-01-06   AAPL  123.123840  126.599998  131.050003  126.379997   
3 2021-01-07   AAPL  127.325241  130.919998  131.630005  127.860001   
4 2021-01-08   AAPL  128.424240  132.050003  132.630005  130.229996   

         open     volume      sector              industry  ...  short_name  \
0  133.520004  143301900  Technology  Consumer Electronics  ...  Apple Inc.   
1  128.889999   97664900  Technology  Consumer Electronics  ...  Apple Inc.   
2  127.720001  155088000  Technology  Consumer Electronics  ...  Apple Inc.   
3  128.360001  109578200  Technology  Consumer Electronics  ...  Apple Inc.   
4  132.429993  105158200  Technology  Consumer Electronics  ...  Apple Inc.   

  currency log_ret_1d  adj_close_ret_1d     ma_5  ma_20  volume_ma_20  \
0      USD        NaN    

In [14]:
print(features.columns)

Index(['date', 'ticker', 'adj_close', 'close', 'high', 'low', 'open', 'volume',
       'sector', 'industry', 'market_cap', 'short_name', 'currency',
       'log_ret_1d', 'adj_close_ret_1d', 'ma_5', 'ma_20', 'volume_ma_20',
       'volume_norm', 'roll_vol_20', 'rsi_14'],
      dtype='str')


In [15]:
print(features.dtypes)

date                datetime64[ms]
ticker                         str
adj_close                  float64
close                      float64
high                       float64
low                        float64
open                       float64
volume                       int64
sector                         str
industry                       str
market_cap                   int64
short_name                     str
currency                       str
log_ret_1d                 float64
adj_close_ret_1d           float64
ma_5                       float64
ma_20                      float64
volume_ma_20               float64
volume_norm                float64
roll_vol_20                float64
rsi_14                     float64
dtype: object


In [16]:
print(features["ticker"].nunique())

88


In [17]:
# What additional features were added which were not in the base panel?
base_columns = set(interim.columns)
feature_columns = set(features.columns)
new_features = feature_columns - base_columns
print(f"New features added: {new_features}")

New features added: {'adj_close_ret_1d', 'log_ret_1d', 'ma_5', 'rsi_14', 'volume_norm', 'volume_ma_20', 'ma_20', 'roll_vol_20'}


# 5. Targets Computation

In [18]:
targets = pd.read_parquet(PROCESSED_DIR / "panel_with_targets.parquet")
print(targets.head())

        date ticker   adj_close       close        high         low  \
0 2021-01-04   AAPL  125.856697  129.410004  133.610001  126.760002   
1 2021-01-05   AAPL  127.412758  131.009995  131.740005  128.429993   
2 2021-01-06   AAPL  123.123840  126.599998  131.050003  126.379997   
3 2021-01-07   AAPL  127.325241  130.919998  131.630005  127.860001   
4 2021-01-08   AAPL  128.424240  132.050003  132.630005  130.229996   

         open     volume      sector              industry  ...  log_ret_1d  \
0  133.520004  143301900  Technology  Consumer Electronics  ...         NaN   
1  128.889999   97664900  Technology  Consumer Electronics  ...    0.012288   
2  127.720001  155088000  Technology  Consumer Electronics  ...   -0.034241   
3  128.360001  109578200  Technology  Consumer Electronics  ...    0.033554   
4  132.429993  105158200  Technology  Consumer Electronics  ...    0.008594   

  adj_close_ret_1d     ma_5  ma_20  volume_ma_20  volume_norm  roll_vol_20  \
0              NaN  

In [19]:
print(f"Base panel columns: {interim.columns}")
print(f"Features panel columns: {features.columns}")
print(f"Targets panel columns: {targets.columns}")

Base panel columns: Index(['date', 'ticker', 'adj_close', 'close', 'high', 'low', 'open', 'volume',
       'sector', 'industry', 'market_cap', 'short_name', 'currency'],
      dtype='str')
Features panel columns: Index(['date', 'ticker', 'adj_close', 'close', 'high', 'low', 'open', 'volume',
       'sector', 'industry', 'market_cap', 'short_name', 'currency',
       'log_ret_1d', 'adj_close_ret_1d', 'ma_5', 'ma_20', 'volume_ma_20',
       'volume_norm', 'roll_vol_20', 'rsi_14'],
      dtype='str')
Targets panel columns: Index(['date', 'ticker', 'adj_close', 'close', 'high', 'low', 'open', 'volume',
       'sector', 'industry', 'market_cap', 'short_name', 'currency',
       'log_ret_1d', 'adj_close_ret_1d', 'ma_5', 'ma_20', 'volume_ma_20',
       'volume_norm', 'roll_vol_20', 'rsi_14', 'future_log_ret_5d',
       'target_class'],
      dtype='str')


In [20]:
# What new columns were added in each change?
features_added = feature_columns - base_columns
targets_added = set(targets.columns) - feature_columns
print(f"Features added: {features_added}")
print(f"Targets added: {targets_added}")

Features added: {'adj_close_ret_1d', 'log_ret_1d', 'ma_5', 'rsi_14', 'volume_norm', 'volume_ma_20', 'ma_20', 'roll_vol_20'}
Targets added: {'future_log_ret_5d', 'target_class'}


In [21]:
# Did any original columns get dropped in the process?
columns_dropped_in_features = base_columns - feature_columns
columns_dropped_in_targets = feature_columns - set(targets.columns)
print(f"Columns dropped in features: {columns_dropped_in_features}")
print(f"Columns dropped in targets: {columns_dropped_in_targets}")

Columns dropped in features: set()
Columns dropped in targets: set()


In [22]:
# Target distribution
print(targets["target_class"].value_counts().sort_index())

# Percentage distribution of target classes
target_distribution = targets["target_class"].value_counts(normalize=True).sort_index() * 100
print(target_distribution)

target_class
-1.0    37212
 0.0    27317
 1.0    45383
Name: count, dtype: int64
target_class
-1.0    33.856176
 0.0    24.853519
 1.0    41.290305
Name: proportion, dtype: float64


# 6. Splits Creation

In [23]:
data_with_splits = pd.read_parquet(PROCESSED_DIR / "panel_with_splits.parquet")
print(data_with_splits.head())

        date ticker   adj_close       close        high         low  \
0 2021-01-04   AAPL  125.856697  129.410004  133.610001  126.760002   
1 2021-01-05   AAPL  127.412758  131.009995  131.740005  128.429993   
2 2021-01-06   AAPL  123.123840  126.599998  131.050003  126.379997   
3 2021-01-07   AAPL  127.325241  130.919998  131.630005  127.860001   
4 2021-01-08   AAPL  128.424240  132.050003  132.630005  130.229996   

         open     volume      sector              industry  ...  \
0  133.520004  143301900  Technology  Consumer Electronics  ...   
1  128.889999   97664900  Technology  Consumer Electronics  ...   
2  127.720001  155088000  Technology  Consumer Electronics  ...   
3  128.360001  109578200  Technology  Consumer Electronics  ...   
4  132.429993  105158200  Technology  Consumer Electronics  ...   

   adj_close_ret_1d     ma_5 ma_20  volume_ma_20  volume_norm  roll_vol_20  \
0               NaN      NaN   NaN           NaN          NaN          NaN   
1          0.0

In [24]:
print(data_with_splits["split"].dtype)

str


In [25]:
# Class counts in each split
split_class_counts = data_with_splits.groupby("split")["target_class"].value_counts().unstack(fill_value=0)
print(split_class_counts)

target_class   -1.0    0.0    1.0
split                            
test           7159   5006   9307
train         22964  16301  26999
val            7089   6010   9077


In [26]:
# Percentages
split_class_percentages = split_class_counts.div(split_class_counts.sum(axis=1), axis=0) * 100
print(split_class_percentages)

target_class       -1.0        0.0        1.0
split                                        
test          33.341095  23.314083  43.344821
train         34.655318  24.600085  40.744597
val           31.966991  27.101371  40.931638


# 7. Tabular Dataset Creation

In [27]:
tabular_dataset = pd.read_parquet(PROCESSED_DIR / "tabular_dataset.parquet")
print(tabular_dataset.head())

  ticker       date  split  target_class      sector     market_cap  \
0   AAPL 2021-03-02  train          -1.0  Technology  3776182222848   
1   AAPL 2021-03-03  train          -1.0  Technology  3776182222848   
2   AAPL 2021-03-04  train           1.0  Technology  3776182222848   
3   AAPL 2021-03-05  train           0.0  Technology  3776182222848   
4   AAPL 2021-03-08  train           1.0  Technology  3776182222848   

   adj_close_mean_20  adj_close_std_20  adj_close_last  adj_close_min_20  ...  \
0         127.460158          5.189738      121.866333        117.843719  ...   
1         126.840277          5.443245      118.885887        117.843719  ...   
2         126.177467          5.800020      117.006081        117.006081  ...   
3         125.409715          5.779573      118.262550        117.006081  ...   
4         124.416241          6.069775      113.334129        113.334129  ...   

   rsi_14_mean_20  rsi_14_std_20  rsi_14_last  rsi_14_min_20  rsi_14_max_20  \
0      

In [28]:
print(tabular_dataset.columns)

Index(['ticker', 'date', 'split', 'target_class', 'sector', 'market_cap',
       'adj_close_mean_20', 'adj_close_std_20', 'adj_close_last',
       'adj_close_min_20', 'adj_close_max_20', 'volume_norm_mean_20',
       'volume_norm_std_20', 'volume_norm_last', 'volume_norm_min_20',
       'volume_norm_max_20', 'log_ret_1d_mean_20', 'log_ret_1d_std_20',
       'log_ret_1d_last', 'log_ret_1d_min_20', 'log_ret_1d_max_20',
       'ma_5_mean_20', 'ma_5_std_20', 'ma_5_last', 'ma_5_min_20',
       'ma_5_max_20', 'ma_20_mean_20', 'ma_20_std_20', 'ma_20_last',
       'ma_20_min_20', 'ma_20_max_20', 'rsi_14_mean_20', 'rsi_14_std_20',
       'rsi_14_last', 'rsi_14_min_20', 'rsi_14_max_20', 'roll_vol_20_mean_20',
       'roll_vol_20_std_20', 'roll_vol_20_last', 'roll_vol_20_min_20',
       'roll_vol_20_max_20'],
      dtype='str')


# 8. Temporal Dataset Creation

In [29]:
temporal_dataset_meta = pd.read_parquet(PROCESSED_DIR / "temporal_index.parquet")
print(temporal_dataset_meta.head())

# We have both the X and y for the temporal dataset in npy files
X_temporal = np.load(PROCESSED_DIR / "X_temporal.npy")
y_temporal = np.load(PROCESSED_DIR / "y_temporal.npy")
print(f"X_temporal shape: {X_temporal.shape}")
print(f"y_temporal shape: {y_temporal.shape}")

# Sample values
print("Sample X_temporal values:")
print(X_temporal[0])
print("Sample y_temporal value:")
print(y_temporal[0])

   sample_id ticker       date  split  target_class
0          0   AAPL 2021-03-02  train            -1
1          1   AAPL 2021-03-03  train            -1
2          2   AAPL 2021-03-04  train             1
3          3   AAPL 2021-03-05  train             0
4          4   AAPL 2021-03-08  train             1
X_temporal shape: (106438, 20, 7)
y_temporal shape: (106438,)
Sample X_temporal values:
[[ 1.31283508e+02  7.28670061e-01  6.31666370e-03  1.00783753e+00
   9.88006413e-01  5.85758171e+01  2.30969973e-02]
 [ 1.30262283e+02  7.88871348e-01 -7.80918822e-03  1.00361383e+00
   9.96845901e-01  5.43508873e+01  2.30680797e-02]
 [ 1.33617599e+02  7.62591481e-01  2.54320037e-02  9.78848517e-01
   9.75740552e-01  6.16100693e+01  2.20941603e-02]
 [ 1.33203598e+02  6.96376860e-01 -3.10321525e-03  9.89198267e-01
   9.80979741e-01  6.39645958e+01  2.10153796e-02]
 [ 1.33349731e+02  6.66306853e-01  1.09646679e-03  9.92453039e-01
   9.81751561e-01  6.34030685e+01  2.09631845e-02]
 [ 1.32473145e+

# 9. Graph Construction

In [30]:
import torch


ticker_to_node_id = pd.read_parquet(PROCESSED_DIR / "ticker_to_node.parquet")
corr_edges = pd.read_parquet(PROCESSED_DIR / "graph_corr_edges.parquet")
js_edges = pd.read_parquet(PROCESSED_DIR / "graph_div_edges.parquet")

edge_index_corr = torch.load(PROCESSED_DIR / "edge_index_corr.pt")
edge_attr_corr = torch.load(PROCESSED_DIR / "edge_attr_corr.pt")
edge_index_js = torch.load(PROCESSED_DIR / "edge_index_js.pt")
edge_attr_js = torch.load(PROCESSED_DIR / "edge_attr_js.pt")

print(ticker_to_node_id.head())
print(corr_edges.head())
print(js_edges.head())

  ticker  node_id
0   AAPL        0
1   ABBV        1
2    ABT        2
3    ACN        3
4   ADBE        4
   src  dst    weight  distance    edge_type
0    0   58  0.861356  0.138644  correlation
1   58    0  0.861356  0.138644  correlation
2    0   37  0.832794  0.167206  correlation
3   37    0  0.832794  0.167206  correlation
4    0   36  0.830874  0.169126  correlation
   src  dst    weight  distance      edge_type
0    0   76  0.979376  0.021059  js_divergence
1   76    0  0.979376  0.021059  js_divergence
2    0   58  0.970300  0.030609  js_divergence
3   58    0  0.970300  0.030609  js_divergence
4    0   27  0.968094  0.032958  js_divergence


In [31]:
# Assess that only 88 unique tickers are present in the ticker_to_node_id mapping
unique_tickers = ticker_to_node_id["ticker"].nunique()
print(f"Number of unique tickers in ticker_to_node_id: {unique_tickers}")

Number of unique tickers in ticker_to_node_id: 88


In [32]:
print(corr_edges["edge_type"].value_counts())

edge_type
correlation    686
sector         258
Name: count, dtype: int64


In [33]:
# Get a random sample where the edge_type is "sector"
sector_edges = corr_edges[corr_edges["edge_type"] == "sector"]
print(sector_edges.sample(1))

     src  dst  weight  distance edge_type
813   83    5     1.0       0.0    sector


In [34]:
# Range for weight column depending on whether it's a correlation edge or a sector edge
corr_edge_weights = corr_edges[corr_edges["edge_type"] == "correlation"]["weight"]
sector_edge_weights = corr_edges[corr_edges["edge_type"] == "sector"]["weight"]

print(f"Correlation edge weights range: {corr_edge_weights.min()} to {corr_edge_weights.max()}")
print(f"Sector edge weights range: {sector_edge_weights.min()} to {sector_edge_weights.max()}")

Correlation edge weights range: 0.6636069436552318 to 0.9969318549184505
Sector edge weights range: 1.0 to 1.0


In [35]:
# Now for the js_edges
print(js_edges["edge_type"].value_counts())

edge_type
js_divergence    622
sector           343
Name: count, dtype: int64


In [36]:
js_edge_weights = js_edges[js_edges["edge_type"] == "js_divergence"]["weight"]
sector_edge_weights = js_edges[js_edges["edge_type"] == "sector"]["weight"]

print(f"JS divergence edge weights range: {js_edge_weights.min()} to {js_edge_weights.max()}")
print(f"Sector edge weights range: {sector_edge_weights.min()} to {sector_edge_weights.max()}")

JS divergence edge weights range: 0.850366748495661 to 0.9872296304112843
Sector edge weights range: 1.0 to 1.0


In [37]:
js_edge_distances = js_edges[js_edges["edge_type"] == "js_divergence"]["distance"]
sector_edge_distances = js_edges[js_edges["edge_type"] == "sector"]["distance"]
print(f"JS divergence edge weights distribution:")
print(js_edge_distances.describe())
print(f"Sector edge weights distribution:")
print(sector_edge_distances.describe())

JS divergence edge weights distribution:
count    622.000000
mean       0.048737
std        0.019036
min        0.012936
25%        0.039713
50%        0.047587
75%        0.054327
max        0.175963
Name: distance, dtype: float64
Sector edge weights distribution:
count    343.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: distance, dtype: float64


In [38]:
# Sample one instance where the edge_type is "js_divergence" and one where it's "sector"
js_divergence_sample = js_edges[js_edges["edge_type"] == "js_divergence"].sample(1)
sector_sample = js_edges[js_edges["edge_type"] == "sector"].sample(1)

print("Sample JS divergence edge:")
print(js_divergence_sample)

print("Sample sector edge:")
print(sector_sample)

Sample JS divergence edge:
     src  dst    weight  distance      edge_type
397   27   46  0.950598  0.051969  js_divergence
Sample sector edge:
     src  dst  weight  distance edge_type
936   42    0     1.0       0.0    sector


# 10. GNN Dataset Creation

In [39]:
snapshot_dataset = pd.read_parquet(PROCESSED_DIR / "gnn_snapshots_index.parquet")
print(snapshot_dataset.head())

X = np.load(PROCESSED_DIR / "X_gnn.npy")
y = np.load(PROCESSED_DIR / "y_gnn.npy")

print(f"X_gnn shape: {X.shape}")
print(f"y_gnn shape: {y.shape}")

   snapshot_id       date  split  num_nodes
0           20 2021-03-02  train         88
1           21 2021-03-03  train         88
2           22 2021-03-04  train         88
3           23 2021-03-05  train         88
4           24 2021-03-08  train         88
X_gnn shape: (1168, 88, 20, 7)
y_gnn shape: (1168, 88)


In [40]:
# Count values for nodes
print(snapshot_dataset["num_nodes"].value_counts())

num_nodes
88    1168
Name: count, dtype: int64


# Sanity Checks

In [41]:
print(X.shape)
print(y.shape)

(1168, 88, 20, 7)
(1168, 88)


- T = 1168 número de días en el training set
- N = 88 número de acciones (nodos)
- W = 20 ventana temporal
- F = 7 número de features por nodo

In [42]:
print(np.isnan(X).sum())
print(np.isnan(y).sum())

0
0


In [49]:
# check values for Y
print(np.unique(y))

[-1  0  1]


Esto será problemático para los modelos, ya que para funciones como CrossEntropy, se espera que los targets sean no-negativos y menores que el número de clases. Por lo tanto, tendremos que aplicar un mapeo a los valores de Y para que se ajusten a este rango. Por ejemplo, podríamos mapear -1 a 0, 0 a 1 y 1 a 2, lo que nos daría tres clases distintas para la clasificación.

In [43]:
print(edge_index_corr.shape)
print(edge_attr_corr.shape)
assert edge_index_corr.shape[1] == edge_attr_corr.shape[0]

torch.Size([2, 944])
torch.Size([944, 5])


In [44]:
print(edge_attr_corr[:, 0].min(), edge_attr_corr[:, 0].max())  # weight
print(edge_attr_corr[:, 1].min(), edge_attr_corr[:, 1].max())  # distance

tensor(0.6636) tensor(1.)
tensor(0.) tensor(0.3364)


## Check de Conectividad 

In [45]:
import networkx as nx

G = nx.Graph()
edges = edge_index_corr.numpy().T
G.add_edges_from(edges)

print("Graph for Correlation Edges:")
print("Num nodes:", G.number_of_nodes())
print("Num edges:", G.number_of_edges())
print("Connected components:", nx.number_connected_components(G))

G = nx.Graph()
G.add_edges_from(edge_index_js.numpy().T)
print("\nGraph for JS Divergence Edges:")
print("Num nodes:", G.number_of_nodes())
print("Num edges:", G.number_of_edges())
print("Connected components:", nx.number_connected_components(G))

Graph for Correlation Edges:
Num nodes: 88
Num edges: 529
Connected components: 1

Graph for JS Divergence Edges:
Num nodes: 88
Num edges: 561
Connected components: 1


Una única componente conexa en ambos grafos y número de nodos y aristas esperado.

#### Distribución de Grados (No queremos demasiados nodos aislados o con grado 1, ni tampoco un nodo con grado muy alto)

In [46]:
degrees = [d for _, d in G.degree()]
print("Min degree:", min(degrees))
print("Max degree:", max(degrees))
print("Mean degree:", np.mean(degrees))

Min degree: 7
Max degree: 24
Mean degree: 12.75


In [47]:
# Alineación de nodos
N = X.shape[1]

assert edge_index_corr.max() < N
assert edge_index_corr.min() >= 0